# 26. Onset-aware training labels and a budget-preserving null

I address two questions on the Bitcoin development period (Binance spot BTCUSDT one-minute data, 2023-01 to 2026-04).

My first question is whether the training label explains the lack of early warning. I trained M0 and M3 with the label "positive if a crash follows in 30–240 minutes", which also counts minutes already in decline as positive. Here I retrain both with the same algorithm, features and budget and change only the label:
- A standard: this label (the notebook 16 model, reproduced exactly).
- B pre-onset only: only minutes before the 1% onset are positive; I drop positive minutes already in decline from training.
- B2 pre-onset only, in-progress negative: as B, but I keep minutes already in decline as negatives (auxiliary).
- C pre-peak only: only minutes before the event peak are positive; I drop positive minutes after the peak from training (strictest).

Peak and onset times enter only the 2021 training labels; model inputs and real-time rules use no future information. I keep training on 2021, threshold calibration on 2022 (no outcome labels, notebook 05 rule) and the budget (20 alerts per month, 240 per year, 240-minute spacing) unchanged.

My second question concerns a budget-preserving null. The regime-matched null moves alerts one at a time and can break the 240-minute spacing and the monthly budget. I add a **budget-preserving regime-matched** null that moves each alert to a minute in the same calendar month and volatility decile, at least 240 minutes from its original position and from alerts already placed, so that monthly counts and the budget stay exactly the same. When the decile has no free minute, I use the nearest decile and record that share. I apply this null to all A, B, B2 and C alerts in the development period and, on the Bitcoin holdout, only to the alerts stored by notebook 11 (post hoc audit, descriptive; I do not evaluate the new models on the holdout).

I pre-specified the reading rules in `26_SPEC.json` before obtaining the results. Label X improves early warning for a method only if (i) its within-regime pre-onset AUC exceeds that of A by at least 0.02 with a paired 95% block-bootstrap interval above 0 and (ii) it has at least 0.5 more above-chance pre-onset catches per 100 alerts than A under the budget-preserving null (a quarter of the development-period M0 value of about 2); it is "worse" if the interval in (i) lies below 0 and otherwise "no material improvement". For the null audit, I call development-period M0 pre-onset warning "robust to the budget constraint" if p < 0.05 under the budget-preserving null, and the holdout reading "unchanged" if its pre-onset p ≥ 0.05.

This analysis is exploratory. I have already examined the development period many times, so an improvement would not be confirmatory; I therefore freeze the A, B, B2 and C models and thresholds with hashes in `ONSET_AWARE_FROZEN.json` for unchanged use in the preregistered prospective test (notebook 31). I write the results to `onset_aware_training_v1/`. The last cells of this saved copy have no output; `ONSET_AWARE_SUMMARY.json` and `4_reading_rule_verdicts.csv` hold the results and verdicts.

In [1]:
from pathlib import Path
from collections import deque, defaultdict
import io, os, sys, json, math, time, hashlib, zipfile, warnings, urllib.request, urllib.error
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])
try:
    from numba import njit
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'      # spot results (notebooks 01–07)
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'            # futures results, read only
BINANCE = 'https://data.binance.vision/data/'
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240; COOLDOWN = 240; COOLDOWN_MIN = 240
B = 240; MONTH_CAP = 20
EVAL_START = '2023-01-01'; EVAL_END = '2026-05-01'

_src = SPOT_BASE / 'cache' / 'DATA_SOURCE.json'
if not _src.is_file() or json.loads(_src.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Spot cache not found: {_src}. Run notebook 01 first.')
REF_PATH = SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json'
THR_PATH = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json'
for _p in (REF_PATH, THR_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 05 and 06 are required: {_p}')
REF = json.loads(REF_PATH.read_text())
FROZEN = json.loads(THR_PATH.read_text())
assert REF.get('venue') == 'spot' and FROZEN.get('venue') == 'spot'
TH_CAL = float(REF['TH_CAL']); TH_HIGH = float(REF['TH_HIGH']); TH_BUCKET = float(FROZEN['TH_BUCKET'])
print('SPOT cache verified. Frozen spot thresholds:', dict(TH_CAL=TH_CAL, TH_HIGH=TH_HIGH, TH_BUCKET=TH_BUCKET))


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def load_btc_spot():
    px = pd.read_parquet(SPOT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet').sort_index()
    sg = pd.read_parquet(SPOT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
    for fr in (px, sg):
        assert isinstance(fr.index, pd.DatetimeIndex) and fr.index.tz is not None
        fr.index = fr.index.tz_convert('UTC')
    assert px.index.equals(sg.index), 'price/signal caches must share one 1-minute index'
    assert px.index.is_unique and (np.diff(px.index.asi8) == NS).all()
    return px, sg


def load_catalog(path):
    E = pd.read_csv(path)
    for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
        E[c] = pd.to_datetime(E[c], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    assert E.event_id.is_unique
    return E


def label_from_catalog(T, E):
    """1 if a 4% breach occurs 30-240 minutes after minute t (same rule as notebooks 04–07)."""
    b = E.breach_utc.astype('int64').to_numpy()
    pos = np.searchsorted(b, T + LEAD_MIN * NS, 'left')
    y = np.zeros(len(T), dtype=np.int8)
    ok = pos < len(b)
    y[ok] = (b[pos[ok]] <= T[ok] + LEAD_MAX * NS).astype(np.int8)
    return y


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def in_period(T, start, end):
    return (T >= ts(start).value) & (T < ts(end).value)


def train_mask(T, valid, year=2021):
    return valid & (T >= ts(f'{year}-01-01').value) & (T < ts(f'{year + 1}-01-01').value - (LEAD_MAX + 5) * NS)


def new_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400,
                                                              solver='lbfgs', random_state=17))


def events_between(E, start, end):
    """Evaluation events: breach in [start+4h, end) so every warning window lies inside the period."""
    return E[(E.breach_utc >= ts(start) + pd.Timedelta(minutes=LEAD_MAX)) & (E.breach_utc < ts(end))].reset_index(drop=True)


# --- emit_calendar, as in notebook 05 ---
def emit_calendar(t,eligible,B=240,cap=20,cooldown_min=240):
    t=np.asarray(t,dtype=np.int64)
    eligible=np.asarray(eligible,dtype=bool)
    assert len(t)==len(eligible) and np.all(np.diff(t)>0)
    ids=np.flatnonzero(eligible)
    z=[]; mcount=0; ycount=0; pm=None; py=None; last=-(10**20)
    for k in ids:
        ti=int(t[k]); dt=pd.Timestamp(ti,tz='UTC')
        year=int(dt.year); month=year*12+int(dt.month)
        if year!=py: ycount=0; py=year
        if month!=pm: mcount=0; pm=month
        if ycount>=B or mcount>=cap or ti-last<cooldown_min*NS:continue
        z.append(ti); last=ti; ycount+=1; mcount+=1
    return np.asarray(z,dtype=np.int64)

# --- match_events, as in notebook 05 ---
def match_events(events,alerts):
    alerts=np.sort(np.asarray(alerts,dtype=np.int64))
    out=[];used=set()
    for ev in events.itertuples(index=False):
        breach=ev.breach_utc.value; onset=ev.onset_1pct_utc.value
        left=int(np.searchsorted(alerts,breach-LEAD_MAX*NS,'left'))
        right=int(np.searchsorted(alerts,breach-LEAD_MIN*NS,'right'))
        for j in range(left,right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id),breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]),tz='UTC'),
                                lead_min=(breach-int(alerts[j]))/NS,
                                pre1pct=bool(alerts[j]<onset)))
                break
    return pd.DataFrame(out,columns=['event_id','breach_utc','alarm_utc','lead_min','pre1pct'])

# --- statistics, as in notebook 05 ---
def statistics(events,alerts):
    hits=match_events(events,alerts)
    n=len(alerts);N=len(events)
    return dict(alarms=n,caught=len(hits),before_1pct=int(hits.pre1pct.sum()),
                false_alarms=n-len(hits),recall=len(hits)/N,
                pre_onset_recall=int(hits.pre1pct.sum())/N,
                alert_precision=(len(hits)/n if n else 0))



def cal_fields(t):
    idx = pd.to_datetime(np.asarray(t, dtype='int64'), utc=True)
    years = idx.year.to_numpy().astype(np.int64)
    months = (idx.year.to_numpy() * 12 + idx.month.to_numpy()).astype(np.int64)
    return years, months


@njit(cache=False)
def _calendar_core(t, elig, years, months, B, caps, cooldown_ns):
    # Line-by-line port of emit_calendar with a per-minute monthly cap (caps) so a surge policy can reuse it.
    out = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B or mcount >= caps[i] or ti - last < cooldown_ns:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1; mcount += 1
    return out[:k].copy()


_CAL_CACHE = [None, None]


def emit_calendar_fast(t, eligible, B=240, cap=20, cooldown_min=240, caps=None):
    t = np.asarray(t, dtype=np.int64); eligible = np.asarray(eligible, dtype=bool)
    assert len(t) == len(eligible) and (len(t) < 2 or np.all(np.diff(t) > 0))
    if _CAL_CACHE[0] is t:
        years, months = _CAL_CACHE[1]
    else:
        years, months = cal_fields(t); _CAL_CACHE[0] = t; _CAL_CACHE[1] = (years, months)
    if caps is None:
        caps = np.full(len(t), int(cap), np.int64)
    return _calendar_core(t, eligible, years, months, int(B), np.asarray(caps, np.int64), np.int64(cooldown_min * NS))


def check_fast_calendar(t, s):
    """Stop if the compiled calendar ever differs from the notebook 05 reference implementation."""
    qs = np.unique(np.quantile(s[np.isfinite(s)], [0., .5, .8, .9, .95, .99, .999]))
    for th in qs:
        e = np.isfinite(s) & (s >= th)
        for B_, cap_ in [(240, 20), (60, 5)]:
            if not np.array_equal(emit_calendar(t, e, B=B_, cap=cap_), emit_calendar_fast(t, e, B=B_, cap=cap_)):
                raise AssertionError(f'fast calendar differs from the reference at th={th}, B={B_}')
    return len(qs) * 2


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate_calendar(t_cal, s_cal, grid=None, B_=240, cap_=20):
    """Notebook 05 rule: the most 2022 alerts reachable (cap = B), then the fewest raw candidates (strictest)."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        e = s_cal >= th
        rows.append((float(th), len(emit_calendar_fast(t_cal, e, B=B_, cap=cap_)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts), df


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p=.5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


def mcnemar_two_sided(a_only, b_only):
    n = a_only + b_only
    return float(binomtest(min(a_only, b_only), n, .5).pvalue) if n else 1.0


def paired(events, base_alerts, new_alerts):
    mb = match_events(events, base_alerts); mn = match_events(events, new_alerts)
    hb, hn = set(mb.event_id), set(mn.event_id)
    pb, pn = set(mb.loc[mb.pre1pct, 'event_id']), set(mn.loc[mn.pre1pct, 'event_id'])
    return dict(base_caught=len(hb), new_caught=len(hn), base_only_caught=len(hb - hn), new_only_caught=len(hn - hb),
                p_caught_two_sided=mcnemar_two_sided(len(hb - hn), len(hn - hb)),
                base_pre1pct=len(pb), new_pre1pct=len(pn), base_only_pre1pct=len(pb - pn), new_only_pre1pct=len(pn - pb),
                p_pre1pct_two_sided=mcnemar_two_sided(len(pb - pn), len(pn - pb)),
                p_pre1pct_one_sided_new_better=mcnemar_one_sided(len(pb - pn), len(pn - pb)))
RAW_COLS = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume',
            'trade_count', 'taker_buy_vol', 'taker_buy_quote', 'ignore']


def month_list(first, last):
    return [p.strftime('%Y-%m') for p in pd.period_range(first, last, freq='M')]


def _fetch(url, tries=5):
    last = None
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            last = e
        except Exception as e:  # network hiccup: retry
            last = e
        time.sleep(2 ** k)
    raise RuntimeError(f'download failed: {url} ({last})')


def get_archives(prefix, symbol, months, dest, allow_missing=False):
    """Binance public monthly 1m klines (prefix 'spot' or 'futures/um'), SHA-256 checked against .CHECKSUM.
    Already-downloaded files whose hash matches are reused, so an interrupted run resumes."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    rows = []
    for ym in months:
        url = f'{BINANCE}{prefix}/monthly/klines/{symbol}/1m/{symbol}-1m-{ym}.zip'
        z = dest / f'{symbol}-1m-{ym}.zip'
        chk = _fetch(url + '.CHECKSUM')
        if chk is None:
            if allow_missing:
                rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=None, status='not_listed'))
                continue
            raise RuntimeError(f'archive not published: {url}')
        expected = chk.decode().split()[0].strip().lower()
        if z.is_file() and sha256_file(z) == expected:
            status = 'cached'
        else:
            data = _fetch(url)
            if data is None:
                raise RuntimeError(f'archive missing although checksum exists: {url}')
            if hashlib.sha256(data).hexdigest() != expected:
                raise RuntimeError(f'SHA-256 mismatch: {url}')
            part = z.parent / (z.name + '.part'); part.write_bytes(data); part.replace(z)
            status = 'downloaded'
        rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=expected, status=status))
    man = pd.DataFrame(rows)
    print(symbol, prefix, man.status.value_counts().to_dict(), flush=True)
    return man


def parse_kline_zip(path):
    """Header or no header; ms or microsecond timestamps (spot archives switched to microseconds in 2025)."""
    with zipfile.ZipFile(path) as zf:
        members = [m for m in zf.namelist() if m.lower().endswith('.csv')]
        if len(members) != 1:
            raise ValueError(f'expected one CSV in {path}')
        with zf.open(members[0]) as fd:
            x = pd.read_csv(fd, header=None, low_memory=False)
    if str(x.iloc[0, 0]).strip().lower() in ('open_time', 'timestamp'):
        x = x.iloc[1:].reset_index(drop=True)
    if x.shape[1] < 12:
        raise ValueError(f'unexpected kline schema in {path}: {x.shape[1]} columns')
    x = x.iloc[:, :12]; x.columns = RAW_COLS
    number = pd.to_numeric(x.open_time, errors='raise')
    mag = float(number.abs().median())
    unit = 'us' if mag > 1e14 else ('ms' if mag > 1e11 else 's')
    z = x[['open', 'high', 'low', 'close', 'volume', 'taker_buy_vol']].apply(pd.to_numeric, errors='coerce')
    z.index = pd.DatetimeIndex(pd.to_datetime(number, unit=unit, utc=True), name='open_time')
    return z[~z.index.duplicated(keep='last')].sort_index().astype('float64')


def load_archives(manifest, folder):
    parts = [parse_kline_zip(Path(folder) / f"{r.symbol}-1m-{r.month}.zip") for r in manifest.itertuples() if r.status != 'not_listed']
    raw = pd.concat(parts).sort_index(kind='stable')
    return raw[~raw.index.duplicated(keep='last')]


def regularize(raw, start=None, end=None):
    """Notebook 01 gap policy: complete UTC minute grid; missing runs of <=5 minutes get forward-filled OHLC and
    zero volume; longer gaps stay NaN."""
    full = pd.date_range(raw.index.min() if start is None else start, raw.index.max() if end is None else end,
                         freq='min', tz='UTC')
    out = raw.reindex(full); out.index.name = 'open_time'
    missing = ~full.isin(raw.index)
    if missing.any():
        changes = np.r_[True, missing[1:] != missing[:-1]]
        grp = np.cumsum(changes) - 1
        runs = np.bincount(grp[missing], minlength=int(grp.max()) + 1)
        short = missing & (runs[grp] <= 5)
        for c in ['open', 'high', 'low', 'close']:
            out.loc[short, c] = out[c].ffill().loc[short]
        for c in ['volume', 'taker_buy_vol']:
            out.loc[short, c] = 0.0
    return out.astype('float64')


# --- chunked_hs, as in notebook 01 ---
def chunked_hs(close,M=32,K=8,chunk=100_000):
    p=close.to_numpy(dtype=np.float32)
    result=np.full(len(p),np.nan,dtype=np.float64)
    for start in range(M-1,len(p),chunk):
        end=min(len(p),start+chunk)
        w=sliding_window_view(p[start-M+1:end],M)
        zero_mean=w-w.mean(axis=1,keepdims=True)
        result[start:end]=np.abs(np.fft.rfft(zero_mean,axis=1))[:,-K:].mean(axis=1)
        if start==(M-1) or end==len(p): print(f'FFT computation {end:,}/{len(p):,}',flush=True)
    return pd.Series(result,index=close.index)



def compute_spot_features(px, with_hs=True):
    """Exactly the notebook 01 signal formulas (z_hs, z_vol, tsr), float32 like the cache."""
    close = px['close']
    r = np.log(close).diff()
    M = 32; rolling_min = 30 * 1440
    out = {}
    if with_hs:
        hs = chunked_hs(close, M=32, K=8)
        log_hs = np.log1p(hs)
        hs_roll = log_hs.rolling(rolling_min, min_periods=rolling_min // 2)
        out['z_hs'] = ((log_hs - hs_roll.mean().shift(1)) / hs_roll.std().shift(1).replace(0, np.nan)).astype('float32')
    rv = r.rolling(M - 1, min_periods=M - 1).std()
    lrv = np.log(rv.where(rv > 0))
    rv_base = lrv.rolling(rolling_min, min_periods=rolling_min // 2)
    out['z_vol'] = ((lrv - rv_base.mean().shift(1)) / rv_base.std().shift(1).replace(0, np.nan)).astype('float32')
    out['tsr'] = ((px.volume - px.taker_buy_vol) / px.volume.replace(0, np.nan)).clip(0, 1).astype('float32')
    return pd.DataFrame(out, index=px.index)


# --- price_only_catalog, as in notebook 07 ---
def price_only_catalog(close,volume,index,H=240,decline=.04,sustain=5,refractory=1440):
    c=np.asarray(close,dtype=np.float64)
    obs=np.isfinite(c)&(c>0)&np.isfinite(volume)&(np.asarray(volume)>0)
    prior=pd.Series(np.where(obs,c,np.nan)).rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=obs & np.isfinite(prior) & (c<=(1-decline)*prior)
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[];last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c) or i-last_confirm<refractory:continue
        threshold=(1-decline)*prior[i]
        if not(obs[i:i+sustain].all() and np.all(c[i:i+sustain]<=threshold)):continue
        if i-H<0 or not obs[i-H:i].all():continue
        prev=c[i-H:i];peak_i=int(i-H+np.argmax(prev))
        if peak_i<=last_confirm:continue
        onset_candidates=np.flatnonzero(c[peak_i+1:i+1]<=.99*c[peak_i])
        if not len(onset_candidates):continue
        oi=int(peak_i+1+onset_candidates[0])
        rows.append({'event_id':len(rows)+1,'peak_utc':index[peak_i],
                     'onset_1pct_utc':index[oi], 'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'peak_close':float(c[peak_i]),'breach_close':float(c[i]),
                     'decline_at_breach_pct':(c[i]/c[peak_i]-1)*100,
                     'peak_to_breach_minutes':i-peak_i,
                     'onset_to_breach_minutes':i-oi})
        last_confirm=i+sustain-1
    df=pd.DataFrame(rows)
    if not len(df):raise RuntimeError('No price-only crash events generated; inspect price history')
    df['year']=pd.DatetimeIndex(df.breach_utc).year
    return df

# Shared with notebook 16 (identical code): VPIN, causal feature set, GBM settings.
REGIME_WINDOW_D = 30; RV_WINDOW = 1440; VPIN_BUCKETS_PER_DAY = 50; VPIN_WINDOW = 50; VPIN_LOOKBACK_D = 30
GBM_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=200, l2_regularization=1.0,
                  class_weight='balanced', early_stopping=False, random_state=17)
FEATS = ['z_vol', 'tsr', 'z_hs', 'ret_5', 'ret_15', 'ret_60', 'ret_240', 'rv_15', 'rv_60', 'rv_240', 'rv_ratio_15_240',
         'dd_240', 'dd_1440', 'vol_intensity_15', 'sell_ratio_15', 'sell_ratio_60', 'vpin']

@njit(cache=False)
def _vpin_core(vol, buy, bucket, n_win):
    out = np.full(len(vol), np.nan); ring = np.zeros(n_win); k = 0; run = 0.0; cv = 0.0; cb = 0.0
    for i in range(len(vol)):
        v = vol[i]; b = buy[i]
        if not (v == v) or not (b == b) or v < 0:
            v = 0.0; b = 0.0
        cv += v; cb += min(max(b, 0.0), v)
        bs = bucket[i]
        if bs == bs and bs > 0 and cv >= bs:
            imb = abs(2.0 * cb - cv) / cv
            j = k % n_win; run += imb - ring[j]; ring[j] = imb; k += 1; cv = 0.0; cb = 0.0
        if k >= n_win:
            out[i] = run / n_win
    return out


def vpin_series(px):
    vol = px.volume.to_numpy(float); buy = px.taker_buy_vol.to_numpy(float)
    daily = px.volume.groupby(px.index.floor('D')).sum()
    size = (daily.rolling(VPIN_LOOKBACK_D, min_periods=VPIN_LOOKBACK_D).mean().shift(1) / VPIN_BUCKETS_PER_DAY)
    bucket = size.reindex(px.index.floor('D')).to_numpy(float)
    return _vpin_core(vol, buy, bucket, VPIN_WINDOW)


def causal_features(px, sg):
    c = px.close.astype(float); lc = np.log(c); r = lc.diff(); vol = px.volume.astype(float); sell = vol - px.taker_buy_vol.astype(float)
    F = pd.DataFrame(index=px.index)
    F['z_vol'] = sg.z_vol.astype(float); F['tsr'] = sg.tsr.astype(float); F['z_hs'] = sg.z_hs.astype(float)
    for k in (5, 15, 60, 240):
        F[f'ret_{k}'] = lc - lc.shift(k)
    for k in (15, 60, 240):
        F[f'rv_{k}'] = r.rolling(k, min_periods=k // 2).std()
    F['rv_ratio_15_240'] = F.rv_15 / F.rv_240
    F['dd_240'] = 1 - c / c.rolling(240, min_periods=120).max()
    F['dd_1440'] = 1 - c / c.rolling(1440, min_periods=720).max()
    F['vol_intensity_15'] = vol.rolling(15, min_periods=8).sum() / (vol.rolling(1440, min_periods=720).sum() / 96)
    F['sell_ratio_15'] = sell.rolling(15, min_periods=8).sum() / vol.rolling(15, min_periods=8).sum()
    F['sell_ratio_60'] = sell.rolling(60, min_periods=30).sum() / vol.rolling(60, min_periods=30).sum()
    F['vpin'] = vpin_series(px)
    return F.replace([np.inf, -np.inf], np.nan).astype('float32')
# Shared by notebooks 18 and 19: exact block-bootstrap AUC decomposition.
def week_pairs(sp, wp, sn, wn):
    """C[k, l] = over positives in block k, the number of negatives in block l with a lower score (+0.5 per tie).
    With block multiplicities w from the bootstrap, the AUC is w'Cw / ((w'P)(w'N)); w = 1 gives the ordinary AUC."""
    C = np.zeros((N_WEEKS, N_WEEKS))
    if len(sp) == 0 or len(sn) == 0:
        return C
    o = np.lexsort((sn, wn)); sn = sn[o]; wn = wn[o]
    bounds = np.searchsorted(wn, np.arange(N_WEEKS + 1), 'left')
    for l in range(N_WEEKS):
        a, b = bounds[l], bounds[l + 1]
        if a == b:
            continue
        seg = sn[a:b]; lo = np.searchsorted(seg, sp, 'left'); hi = np.searchsorted(seg, sp, 'right')
        C[:, l] = np.bincount(wp, weights=lo + 0.5 * (hi - lo), minlength=N_WEEKS)
    return C


def _ratio(a, b):
    return a / b if b > 0 else np.nan


STAT_KEYS = ['auc_standard', 'auc_pre', 'auc_in', 'share_in', 'diff_standard_minus_pre', 'in_share_of_excess', 'auc_calm',
             'auc_standard_within_regime', 'auc_pre_within_regime', 'auc_in_within_regime']


def make_parts(sm, code, calm, wk, dec):
    neg = code == 0; pre = code == 1; inn = code == 2; cnt = lambda k: np.bincount(wk[k], minlength=N_WEEKS).astype(float)
    P = dict(C_pre=week_pairs(sm[pre], wk[pre], sm[neg], wk[neg]), C_in=week_pairs(sm[inn], wk[inn], sm[neg], wk[neg]),
             C_calm=week_pairs(sm[(code > 0) & calm], wk[(code > 0) & calm], sm[neg & calm], wk[neg & calm]),
             P_pre=cnt(pre), P_in=cnt(inn), N=cnt(neg), P_calm=cnt((code > 0) & calm), N_calm=cnt(neg & calm),
             Cs_pre=np.zeros((N_WEEKS, N_WEEKS)), Cs_in=np.zeros((N_WEEKS, N_WEEKS)), Pq_pre=[], Pq_in=[], Nq=[])
    for q in range(10):
        kq = dec == q; nq = neg & kq; pq = pre & kq; iq = inn & kq
        P['Cs_pre'] += week_pairs(sm[pq], wk[pq], sm[nq], wk[nq]); P['Cs_in'] += week_pairs(sm[iq], wk[iq], sm[nq], wk[nq])
        P['Pq_pre'].append(cnt(pq)); P['Pq_in'].append(cnt(iq)); P['Nq'].append(cnt(nq))
    for k in ('Pq_pre', 'Pq_in', 'Nq'):
        P[k] = np.array(P[k])
    return P


def stats_from(P, w):
    q = lambda C: w @ C @ w
    Wn = w @ P['N']; Wp = w @ P['P_pre']; Wi = w @ P['P_in']; sp = q(P['C_pre']); si = q(P['C_in'])
    std = _ratio(sp + si, (Wp + Wi) * Wn); pre = _ratio(sp, Wp * Wn); inn = _ratio(si, Wi * Wn); pi = _ratio(Wi, Wp + Wi)
    nq = P['Nq'] @ w; pq = P['Pq_pre'] @ w; iq = P['Pq_in'] @ w; ssp = q(P['Cs_pre']); ssi = q(P['Cs_in'])
    return dict(auc_standard=std, auc_pre=pre, auc_in=inn, share_in=pi, diff_standard_minus_pre=std - pre,
                in_share_of_excess=_ratio(pi * (inn - 0.5), std - 0.5) if np.isfinite(inn) else 0.0,
                auc_calm=_ratio(q(P['C_calm']), (w @ P['P_calm']) * (w @ P['N_calm'])),
                auc_standard_within_regime=_ratio(ssp + ssi, ((pq + iq) * nq).sum()),
                auc_pre_within_regime=_ratio(ssp, (pq * nq).sum()), auc_in_within_regime=_ratio(ssi, (iq * nq).sum()))
import bisect
import joblib
from sklearn.ensemble import HistGradientBoostingClassifier

OUT = SPOT_BASE / 'onset_aware_training_v1'; (OUT / 'models').mkdir(parents=True, exist_ok=True)
M16 = SPOT_BASE / 'methods_definitions_v1'
M21 = SPOT_BASE / 'onset_positioning_volatility_v1'
HOLD = SPOT_BASE / 'HOLDOUT_2018_2020_confirmatory_v1'
RAW_HOLD = DRIVE / 'BTCUSDT_spot_holdout_2017_2020' / '1m'
DNAME, DCL = '4pct', 0.04
LABELS = ['A_standard', 'B_pre_onset', 'B2_pre_onset_in_decline_negative', 'C_pre_peak']
TRAINED = ['M0_VF_logit', 'M3_GBM_rich']
N_BOOT = 500; N_NULL = 1000; SEED = 20261108; BLOCK_DAYS = 7; BP_TRIES = 64
MIN_AUC_GAIN = 0.02; MIN_EXCESS_GAIN = 0.5
H_START, H_END = '2018-01-01', '2021-01-01'
EVAL_LAST = ts(EVAL_END).value - (LEAD_MAX + 5) * NS

for _p in [M16 / 'METHODS_FROZEN.json', M16 / 'METHODS_SPEC.json', M16 / f'alarms_M0_VF_logit_{DNAME}.csv', M16 / f'alarms_M3_GBM_rich_{DNAME}.csv',
           M16 / '4_methods_definitions_nulls.csv', M21 / '1_onset_per_minute.csv',
           HOLD / 'HOLDOUT_RESULTS.json', HOLD / 'holdout_events.csv', HOLD / 'alarms_calendar.csv', HOLD / 'spot_download_manifest.csv']:
    if not _p.is_file():
        raise FileNotFoundError(f'Results of earlier notebooks (11, 16, 21) are required: {_p}')
FROZEN16 = json.loads((M16 / 'METHODS_FROZEN.json').read_text()); SPEC16 = json.loads((M16 / 'METHODS_SPEC.json').read_text())
if sha256_file(M16 / 'METHODS_SPEC.json') != FROZEN16['spec_sha256'] or SPEC16['gbm_params'] != GBM_PARAMS or SPEC16['gbm_features'] != FEATS:
    raise RuntimeError('GUARD FAIL: notebook-16 spec/settings differ')

SPEC = {'title': 'Onset-aware training labels and a budget-preserving null (dev 2023-01..2026-04, exploratory)',
        'labels': {'A_standard': 'positive if a 4% breach occurs 30-240 min later (notebook 16)',
                   'B_pre_onset': 'positive only if also before the 1% onset of that event; positive minutes at or after the onset are dropped from training',
                   'B2_pre_onset_in_decline_negative': 'as B, but positive minutes at or after the onset are kept as negatives',
                   'C_pre_peak': 'positive only if also before the peak of that event; positive minutes at or after the peak are dropped from training'},
        'label_use': 'peak and onset times enter only the 2021 training labels; features and alert rules are causal',
        'methods': {'M0_VF_logit': 'logistic on z_vol and tsr (notebook 16)', 'M3_GBM_rich': f'HistGradientBoosting, notebook-16 parameters, {len(FEATS)} features'},
        'training': '2021 minus last 245 min', 'calibration': '2022, notebook-5 calendar rule, 20/month, 240/yr, 240-min cooldown, no outcome labels',
        'evaluation': {'alerts': 'pre-onset, in-progress, pre-peak and lead >= 30/60 min before onset, dev 2023-01..2026-04',
                       'per_minute': f'notebook-18/21 decomposition (L1 label, event split); paired differences vs A from one shared {N_BOOT} x {BLOCK_DAYS}-day block bootstrap'},
        'nulls': {'circular': '>= 7 days, all alerts shifted together', 'regime': 'each alert to a random minute of the same 24h-RV decile within +-30 days, >= 240 min away',
                  'budget_preserving_regime': ('each alert, in random order, to a random minute of the same calendar month and the same 24h-RV decile, '
                                               '>= 240 min from its original and >= 240 min from every alert already placed; nearest decile if none fits; '
                                               f'{BP_TRIES} candidates per decile; monthly counts are identical by construction'),
                  'draws': N_NULL},
        'reading_rule': (f'Label X improves early warning for method m only if (i) within-regime pre-onset AUC (X minus A) is at least {MIN_AUC_GAIN} '
                         'and its paired 95% block-bootstrap interval lies above 0, and (ii) X has at least '
                         f'{MIN_EXCESS_GAIN} more above-chance pre-onset catches per 100 alerts than A under the budget-preserving null. '
                         '"Worse" if the interval in (i) lies below 0; otherwise "no material improvement". Reported for every m and X; nothing is selected.'),
        'null_audit_rule': ('Dev M0 pre-onset warning is called robust to the budget constraint if p < 0.05 under the budget-preserving null; '
                            'the Bitcoin holdout reading is called unchanged if its pre-onset p >= 0.05 under that null.'),
        'holdout_use': 'notebook-11 stored alerts only, re-scored under the nulls (post hoc, descriptive); no new model touches any holdout',
        'prospective': 'B, B2 and C models and thresholds are frozen with SHA-256 hashes for a pre-registered test on data from November 2026 onward',
        'seed': SEED, 'notebook16_spec_sha256': FROZEN16['spec_sha256']}
_raw = (json.dumps(SPEC, indent=2, sort_keys=True, default=str) + '\n').encode()
_sp = OUT / '26_SPEC.json'
if _sp.exists() and _sp.read_bytes() != _raw:
    _sp.rename(OUT / f'26_SPEC.replaced_{pd.Timestamp.now(tz="UTC"):%Y%m%dT%H%M%S}.json')
_sp.write_bytes(_raw); SPEC_SHA = hashlib.sha256(_raw).hexdigest()
print('Output:', OUT, '| spec sha256', SPEC_SHA[:16])

Mounted at /content/drive
SPOT cache verified. Frozen spot thresholds: {'TH_CAL': 0.6698990591069133, 'TH_HIGH': 0.6864782740093911, 'TH_BUCKET': 0.7166067355142407}
Output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/onset_aware_training_v1 | spec sha256 b061b78cb73807b0


## 1. Data, features, events and label variants

In [2]:
t_ = time.time()
px, sg = load_btc_spot()
F = causal_features(px, sg)
T = px.index.asi8; close = px.close.to_numpy(float); volume = px.volume.to_numpy(float)
v_ = sg.z_vol.to_numpy(float); f_ = sg.tsr.to_numpy(float); h_ = sg.z_hs.to_numpy(float)
VALID = (np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume > 0) & np.isfinite(v_) & np.isfinite(f_) & np.isfinite(h_) & (f_ >= 0) & (f_ <= 1))
CAL = VALID & in_period(T, '2022-01-01', '2023-01-01'); EVA = VALID & in_period(T, EVAL_START, EVAL_END)
EVM = VALID & (T >= ts(EVAL_START).value) & (T < EVAL_LAST)
DD240 = F.dd_240.to_numpy(float); CALM = np.isfinite(DD240) & (DD240 < DCL / 4)
_lc = np.log(np.where(np.isfinite(close) & (close > 0), close, np.nan))
RV = pd.Series(_lc).diff().rolling(RV_WINDOW, min_periods=RV_WINDOW // 2).std().to_numpy()
DEC = np.full(len(T), -1, np.int64); _okr = EVM & np.isfinite(RV)
DEC[_okr] = np.minimum((pd.Series(RV[_okr]).rank(pct=True).to_numpy() * 10).astype(int), 9)
WEEK = np.where(EVM, (T - ts(EVAL_START).value) // (BLOCK_DAYS * 1440 * NS), -1).astype(np.int64)
N_WEEKS = int(WEEK[EVM].max()) + 1

E = load_catalog(SPOT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv')
EVD = events_between(E, EVAL_START, EVAL_END)
Y = label_from_catalog(T, E)
_br = E.breach_utc.astype('int64').to_numpy(); _on = E.onset_1pct_utc.astype('int64').to_numpy(); _pk = E.peak_utc.astype('int64').to_numpy()
_pos = np.searchsorted(_br, T + LEAD_MIN * NS, 'left'); _okp = _pos < len(_br)
INN = np.zeros(len(T), bool); INN[_okp] = T[_okp] >= _on[_pos[_okp]]; INN &= Y == 1          # positive and already in decline
PREPK = np.zeros(len(T), bool); PREPK[_okp] = T[_okp] < _pk[_pos[_okp]]; PREPK &= Y == 1     # positive and before the event peak
TR = train_mask(T, VALID)


def label_variant(name):
    """(labels, training mask) for one variant; only the 2021 training rows are affected."""
    if name == 'A_standard':
        return Y, TR
    if name == 'B_pre_onset':
        return Y, TR & ~INN
    if name == 'B2_pre_onset_in_decline_negative':
        return (Y.astype(bool) & ~INN).astype(np.int8), TR
    if name == 'C_pre_peak':
        return Y, TR & ~(Y.astype(bool) & ~PREPK)
    raise KeyError(name)


lc = []
for lab in LABELS:
    yv, use = label_variant(lab)
    lc.append(dict(label=lab, train_minutes=int(use.sum()), positives=int(yv[use].sum()), negatives=int((yv[use] == 0).sum()),
                   dropped_vs_A=int(TR.sum() - use.sum()), positive_share=float(yv[use].mean())))
LC = pd.DataFrame(lc); LC.to_csv(OUT / '0_training_label_counts.csv', index=False); display(LC)
print(f'data ready ({time.time() - t_:.0f}s); dev events {len(EVD)}')

,label,train_minutes,positives,negatives,dropped_vs_A,positive_share
0,A_standard,524087,18779,505308,0,0.035832
1,B_pre_onset,513412,8104,505308,10675,0.015785
2,B2_pre_onset_in_decline_negative,524087,8104,515983,0,0.015463
3,C_pre_peak,510665,5357,505308,13422,0.010490


data ready (22s); dev events 83


## 2. Training, 2022 calibration and development-period alerts

Label A reproduces the notebook 16 M0 and M3 models exactly. For M3 A I use the notebook 16 model file as is and record whether a refit with the same settings gives the same scores, so that I can attribute differences from B and C to the label.

In [ ]:
Xl = F[['z_vol', 'tsr']].to_numpy(float); XL_OK = np.isfinite(Xl).all(axis=1)
X_all = F[FEATS].to_numpy(np.float32)
SC = {}; ALR = {}; frozen = {'spec_sha256': SPEC_SHA, 'configs': {}}; t_ = time.time()
for mname in TRAINED:
    cfg16 = FROZEN16['configs'][f'{mname}|{DNAME}']
    a16 = np.sort(pd.to_datetime(pd.read_csv(M16 / f'alarms_{mname}_{DNAME}.csv').alarm_utc, utc=True).astype('int64').to_numpy())
    for lab in LABELS:
        t0 = time.time(); yv, use = label_variant(lab); s = np.full(len(T), np.nan); info = {}
        if mname == 'M0_VF_logit':
            mk = use & XL_OK
            mdl = new_logit().fit(Xl[mk], yv[mk]); okp = VALID & XL_OK; s[okp] = mdl.predict_proba(Xl[okp])[:, 1]
            info = dict(coef=mdl[-1].coef_.ravel().tolist(), intercept=float(mdl[-1].intercept_[0]),
                        scaler_mean=mdl[0].mean_.tolist(), scaler_scale=mdl[0].scale_.tolist())
            if lab == 'A_standard' and not (np.allclose(info['coef'], cfg16['coef'], rtol=1e-6, atol=1e-9)
                                            and np.isclose(info['intercept'], cfg16['intercept'], rtol=1e-6, atol=1e-9)):
                raise RuntimeError('GUARD FAIL: M0 refit differs from notebook 16')
        else:
            if lab == 'A_standard':
                mp = SPOT_BASE / cfg16['model_file']
                if sha256_file(mp) != cfg16['model_sha256']:
                    raise RuntimeError('GUARD FAIL: notebook-16 GBM file SHA-256 differs')
                mdl = joblib.load(mp)
                chk = HistGradientBoostingClassifier(**GBM_PARAMS).fit(X_all[use], yv[use])
                smp = np.flatnonzero(EVA)[::max(1, int(EVA.sum()) // 20000)]
                refit_diff = float(np.abs(chk.predict_proba(X_all[smp])[:, 1] - mdl.predict_proba(X_all[smp])[:, 1]).max())
                print(f'M3 A: refit vs notebook-16 file, max |score difference| on {len(smp)} dev minutes = {refit_diff:.2e}'
                      + ('' if refit_diff < 1e-9 else '  (warning: refit is not bit-identical; B/C differences then include refit noise)'))
                info = dict(model_file=cfg16['model_file'], model_sha256=cfg16['model_sha256'], refit_max_abs_diff=refit_diff)
                del chk
            else:
                mdl = HistGradientBoostingClassifier(**GBM_PARAMS).fit(X_all[use], yv[use])
                mp = OUT / 'models' / f'{mname}_{lab}.joblib'; joblib.dump(mdl, mp)
                info = dict(model_file=str(mp.relative_to(SPOT_BASE)), model_sha256=sha256_file(mp))
            s[VALID] = mdl.predict_proba(X_all[VALID])[:, 1]
        okc = CAL & np.isfinite(s); oke = EVA & np.isfinite(s)
        th, ncal, _ = calibrate_calendar(T[okc], s[okc])
        a = emit_calendar_fast(T[oke], s[oke] >= th)
        if lab == 'A_standard' and (abs(th - cfg16['threshold_2022']) > 1e-12 or not np.array_equal(a, a16)):
            raise RuntimeError(f'GUARD FAIL: {mname} A does not reproduce notebook-16 threshold/alerts')
        SC[(mname, lab)] = s; ALR[(mname, lab)] = a
        pd.DataFrame({'alarm_utc': pd.to_datetime(a, utc=True)}).to_csv(OUT / f'alarms_{mname}_{lab}.csv', index=False)
        frozen['configs'][f'{mname}|{lab}'] = dict(method=mname, label=lab, threshold_2022=th, cal_alerts_2022=ncal, dev_alerts=len(a), **info)
        print(f'{mname:12s} {lab:34s} threshold {th:.6g} | 2022 alerts {ncal} | dev alerts {len(a)} ({time.time() - t0:.0f}s)', flush=True)
(OUT / 'ONSET_AWARE_FROZEN.json').write_text(json.dumps(frozen, indent=2, default=float))
FROZEN_SHA = sha256_file(OUT / 'ONSET_AWARE_FROZEN.json')
print('PASS: A reproduces notebook 16 for M0 and M3 | ONSET_AWARE_FROZEN.json sha256', FROZEN_SHA, f'({time.time() - t_:.0f}s)')

M0_VF_logit  A_standard                         threshold 0.669899 | 2022 alerts 240 | dev alerts 800 (4s)
M0_VF_logit  B_pre_onset                        threshold 0.59639 | 2022 alerts 240 | dev alerts 800 (3s)
M0_VF_logit  B2_pre_onset_in_decline_negative   threshold 0.589926 | 2022 alerts 240 | dev alerts 800 (2s)
M0_VF_logit  C_pre_peak                         threshold 0.625673 | 2022 alerts 240 | dev alerts 800 (2s)
M3 A: refit vs notebook-16 file, max |score difference| on 20125 dev minutes = 0.00e+00
M3_GBM_rich  A_standard                         threshold 0.113531 | 2022 alerts 240 | dev alerts 768 (124s)
M3_GBM_rich  B_pre_onset                        threshold 0.0910182 | 2022 alerts 240 | dev alerts 627 (125s)
M3_GBM_rich  B2_pre_onset_in_decline_negative   threshold 0.0955868 | 2022 alerts 240 | dev alerts 643 (128s)
M3_GBM_rich  C_pre_peak                         threshold 0.114367 | 2022 alerts 240 | dev alerts 580 (131s)
PASS: A reproduces notebook 16 for M0 and M3 | 

## 3. Nulls: circular shift, regime-matched, budget-preserving regime-matched

I match alerts to events with the notebook 05 rule (for each event, the earliest unused alert 30–240 minutes before the crash).

In [ ]:
def ev_arrays(ev):
    return dict(br=ev.breach_utc.astype('int64').to_numpy(), on=ev.onset_1pct_utc.astype('int64').to_numpy(),
                pk=ev.peak_utc.astype('int64').to_numpy(), n=len(ev))


def fast_counts(a, ea):
    a = np.sort(np.asarray(a, np.int64)); used = np.zeros(len(a), bool); pre = post = pk = l30 = l60 = 0
    for k in range(ea['n']):
        b = ea['br'][k]; l_ = np.searchsorted(a, b - LEAD_MAX * NS, 'left'); r_ = np.searchsorted(a, b - LEAD_MIN * NS, 'right')
        for j in range(l_, r_):
            if not used[j]:
                used[j] = True; t = a[j]; o = ea['on'][k]
                if t < o:
                    pre += 1; pk += int(t < ea['pk'][k]); l30 += int(t <= o - 30 * NS); l60 += int(t <= o - 60 * NS)
                else:
                    post += 1
                break
    return dict(pre=pre, post=post, pre_peak=pk, pre_lead30=l30, pre_lead60=l60)


CKEYS = ['pre', 'post', 'pre_peak', 'pre_lead30', 'pre_lead60']
DQ_ORDER = [sorted(range(10), key=lambda d, q=q: (abs(d - q), d)) for q in range(10)]


def make_space(Tp, RVp, start, end):
    okd = (Tp >= ts(start).value) & (Tp < ts(end).value) & np.isfinite(RVp)
    tt = Tp[okd]; dq = np.minimum((pd.Series(RVp[okd]).rank(pct=True).to_numpy() * 10).astype(int), 9)
    o = np.lexsort((tt, dq)); ds = dq[o]
    mo = cal_fields(tt)[1]; key = mo * 10 + dq; o2 = np.lexsort((tt, key))
    return dict(t0=ts(start).value, L=(ts(end).value - ts(start).value) // NS, TT=tt, DD=dq, PP=tt[o],
                STARTS=np.searchsorted(ds, np.arange(10), 'left'), ENDS=np.searchsorted(ds, np.arange(10), 'right'),
                PK=tt[o2], KS=key[o2])


def draw_circular(a, rng, sp):
    d = int(rng.integers(7 * 1440, sp['L'] - 7 * 1440))
    return np.sort(sp['t0'] + (a - sp['t0'] + d * NS) % (sp['L'] * NS))


def draw_regime(a, rng, sp, W=30 * 1440 * NS):
    TT_, PP_, STARTS_, ENDS_ = sp['TT'], sp['PP'], sp['STARTS'], sp['ENDS']
    j = np.clip(np.searchsorted(TT_, a, 'right') - 1, 0, len(TT_) - 1); ad = sp['DD'][j]
    lo = np.array([STARTS_[q] + np.searchsorted(PP_[STARTS_[q]:ENDS_[q]], ti - W, 'left') for ti, q in zip(a, ad)])
    hi = np.array([STARTS_[q] + np.searchsorted(PP_[STARTS_[q]:ENDS_[q]], ti + W, 'right') for ti, q in zip(a, ad)])
    c = PP_[lo + (rng.random(len(a)) * (hi - lo)).astype(np.int64)]; bad = np.abs(c - a) < LEAD_MAX * NS
    for _ in range(20):
        if not bad.any():
            break
        c[bad] = PP_[lo[bad] + (rng.random(int(bad.sum())) * (hi[bad] - lo[bad])).astype(np.int64)]; bad = np.abs(c - a) < LEAD_MAX * NS
    return np.sort(c)


def draw_budget(a, rng, sp, audit=None):
    """Budget-preserving regime-matched draw: same calendar month, same (or nearest) RV decile, >= 240 min from the original
    and from every alert already placed. Alerts that cannot be placed keep their original time (conservative) and are counted."""
    a = np.asarray(a, np.int64); j = np.clip(np.searchsorted(sp['TT'], a, 'right') - 1, 0, len(sp['TT']) - 1)
    q0 = sp['DD'][j]; mo = cal_fields(a)[1]; KS, PK = sp['KS'], sp['PK']
    placed = []; out = np.empty(len(a), np.int64); exact = near = failed = 0
    for i in rng.permutation(len(a)):
        ai = int(a[i]); done = False
        for dq in DQ_ORDER[q0[i]]:
            key = mo[i] * 10 + dq; lo = np.searchsorted(KS, key, 'left'); hi = np.searchsorted(KS, key, 'right')
            if hi <= lo:
                continue
            for c in PK[lo + rng.integers(0, hi - lo, BP_TRIES)]:
                c = int(c)
                if abs(c - ai) < LEAD_MAX * NS:
                    continue
                k = bisect.bisect_left(placed, c)
                if (k > 0 and c - placed[k - 1] < COOLDOWN * NS) or (k < len(placed) and placed[k] - c < COOLDOWN * NS):
                    continue
                placed.insert(k, c); out[i] = c; done = True; break
            if done:
                exact += int(dq == q0[i]); near += int(dq != q0[i]); break
        if not done:
            failed += 1; out[i] = ai; bisect.insort(placed, ai)
    if audit is not None:
        audit['exact'] += exact; audit['near'] += near; audit['failed'] += failed; audit['n'] += len(a)
    return np.sort(out)


def constraint_check(a_null, a_ref):
    """Monthly counts vs the real alerts, months over 20, years over 240, gaps under 240 minutes."""
    mr = pd.Series(cal_fields(a_ref)[1]).value_counts(); mn = pd.Series(cal_fields(a_null)[1]).value_counts()
    yn = pd.Series(cal_fields(a_null)[0]).value_counts()
    return dict(months_count_changed=int((mn.reindex(mr.index.union(mn.index), fill_value=0) != mr.reindex(mr.index.union(mn.index), fill_value=0)).sum()),
                months_over_cap=int((mn > MONTH_CAP).sum()), years_over_budget=int((yn > B).sum()),
                gaps_under_cooldown=int((np.diff(np.sort(a_null)) < COOLDOWN * NS).sum()))


NULLS = [('circular >=7d', draw_circular), ('regime-matched', draw_regime), ('budget-preserving regime', draw_budget)]
DSP = make_space(T, RV, EVAL_START, EVAL_END); EAD = ev_arrays(EVD)
_a = ALR[('M0_VF_logit', 'A_standard')]; _m = match_events(EVD, _a); _f = fast_counts(_a, EAD)
if (_f['pre'], _f['post']) != (int(_m.pre1pct.sum()), int((~_m.pre1pct).sum())):
    raise RuntimeError('GUARD FAIL: fast matcher differs from match_events')
R16 = pd.read_csv(M16 / '4_methods_definitions_nulls.csv')
for mname in TRAINED:
    r16 = R16[(R16.method == mname) & (R16.definition == DNAME)].iloc[0]; f_ = fast_counts(ALR[(mname, 'A_standard')], EAD)
    if (f_['pre'], f_['post']) != (int(r16.pre), int(r16.post)):
        raise RuntimeError(f'GUARD FAIL: {mname} A counts differ from notebook 16')
print('PASS: fast matcher equals match_events; A counts equal notebook 16')


def run_nulls(alert_sets, ea, sp, seed0, tag):
    rows, audit_rows = [], []
    for si, (key, a) in enumerate(alert_sets.items()):
        obs = fast_counts(a, ea)
        for ni, (nname, drawer) in enumerate(NULLS):
            rng = np.random.default_rng(seed0 + 100 * si + ni); dist = {k: np.empty(N_NULL) for k in CKEYS}
            aud = dict(exact=0, near=0, failed=0, n=0); cc = []
            for i in range(N_NULL):
                sh = drawer(a, rng, sp, aud) if drawer is draw_budget else drawer(a, rng, sp)
                c = fast_counts(sh, ea)
                for k in CKEYS:
                    dist[k][i] = c[k]
                if i < 50:
                    cc.append(constraint_check(sh, a))
            row = dict(period=tag, set=key[0], label=key[1], null=nname, alerts=len(a), events=ea['n'])
            for k in CKEYS:
                row[k] = obs[k]; row[f'{k}_null_mean'] = float(dist[k].mean())
                row[f'p_{k}'] = (1 + int((dist[k] >= obs[k]).sum())) / (N_NULL + 1)
                row[f'{k}_excess_per_100_alerts'] = 100 * (obs[k] - float(dist[k].mean())) / len(a)
            rows.append(row)
            ccd = pd.DataFrame(cc)
            audit_rows.append(dict(period=tag, set=key[0], label=key[1], null=nname,
                                   share_same_decile=(aud['exact'] / aud['n']) if aud['n'] else np.nan,
                                   share_nearest_decile=(aud['near'] / aud['n']) if aud['n'] else np.nan,
                                   share_unplaced=(aud['failed'] / aud['n']) if aud['n'] else np.nan,
                                   **{f'{k}_mean_first50': float(ccd[k].mean()) for k in ccd.columns}))
        print(f'{tag} {key[0]} {key[1]}: nulls done', flush=True)
    return rows, audit_rows


t_ = time.time()
rows, audit_rows = run_nulls({k: ALR[k] for k in ALR}, EAD, DSP, SEED, 'dev')
print(f'dev nulls done ({time.time() - t_:.0f}s)')

PASS: fast matcher equals match_events; A counts equal notebook 16
dev M0_VF_logit A_standard: nulls done
dev M0_VF_logit B_pre_onset: nulls done
dev M0_VF_logit B2_pre_onset_in_decline_negative: nulls done
dev M0_VF_logit C_pre_peak: nulls done
dev M3_GBM_rich A_standard: nulls done
dev M3_GBM_rich B_pre_onset: nulls done
dev M3_GBM_rich B2_pre_onset_in_decline_negative: nulls done
dev M3_GBM_rich C_pre_peak: nulls done
dev nulls done (273s)


## 4. Bitcoin holdout: notebook 11 stored alerts rescored under the budget-preserving null (post hoc audit)

From the holdout prices I compute only the volatility deciles, with the same method as notebook 12. This audit leaves the hypotheses and results of notebook 11 unchanged.

In [ ]:
TCOLS = ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']
hev = pd.read_csv(HOLD / 'holdout_events.csv')
for c_ in TCOLS:
    hev[c_] = pd.to_datetime(hev[c_], utc=True)
hev = hev.sort_values('breach_utc').reset_index(drop=True)
ah = np.sort(pd.to_datetime(pd.read_csv(HOLD / 'alarms_calendar.csv').alarm_utc, utc=True).astype('int64').to_numpy())
H1 = json.loads((HOLD / 'HOLDOUT_RESULTS.json').read_text())['H1']
sth = statistics(hev, ah)
if [sth['alarms'], sth['caught'], sth['before_1pct']] != [H1['alarms'], H1['any_hits'], H1['pre_onset_hits']]:
    raise RuntimeError('GUARD FAIL: saved holdout alerts/events do not reproduce notebook 11')
man = pd.read_csv(HOLD / 'spot_download_manifest.csv')
for r in man.itertuples():
    if r.status != 'not_listed' and sha256_file(RAW_HOLD / f'{r.symbol}-1m-{r.month}.zip') != r.sha256:
        raise RuntimeError(f'GUARD FAIL: holdout archive {r.month} SHA-256 differs from the notebook-11 manifest')
Hh = regularize(load_archives(man, RAW_HOLD), end=pd.Timestamp('2020-12-31 23:59', tz='UTC'))
Hh = Hh.loc[:pd.Timestamp('2020-12-31 23:59', tz='UTC')]
Th = Hh.index.asi8; RVh = pd.Series(np.log(Hh.close.to_numpy(float))).diff().rolling(RV_WINDOW, min_periods=RV_WINDOW // 2).std().to_numpy()
del Hh
HSP = make_space(Th, RVh, H_START, H_END); EAH = ev_arrays(hev)
fh = fast_counts(ah, EAH)
if (fh['pre'], fh['pre'] + fh['post']) != (H1['pre_onset_hits'], H1['any_hits']):
    raise RuntimeError('GUARD FAIL: fast matcher differs on the holdout')
print(f'PASS holdout: notebook-11 alerts/events reproduced ({fh["pre"]} pre-onset, {fh["post"]} in-progress)')
t_ = time.time()
r_h, a_h = run_nulls({('M0_VF_logit', 'A_standard (notebook 11 stored)'): ah}, EAH, HSP, SEED + 9000, 'holdout')
rows += r_h; audit_rows += a_h
AL = pd.DataFrame(rows); AL.to_csv(OUT / '1_alerts_by_label_and_null.csv', index=False)
AU = pd.DataFrame(audit_rows); AU.to_csv(OUT / '2_null_constraint_audit.csv', index=False)
print(f'holdout nulls done ({time.time() - t_:.0f}s)')
display(AU.round(4))
display(AL.pivot_table(index=['period', 'set', 'label'], columns='null', values=['pre', 'pre_excess_per_100_alerts', 'p_pre'], sort=False).round(3))
display(AL.pivot_table(index=['period', 'set', 'label'], columns='null', values=['post', 'post_excess_per_100_alerts', 'p_post'], sort=False).round(3))

PASS holdout: notebook-11 alerts/events reproduced (13 pre-onset, 23 in-progress)
holdout M0_VF_logit A_standard (notebook 11 stored): nulls done
holdout nulls done (37s)


,period,set,label,null,share_same_decile,share_nearest_decile,share_unplaced,months_count_changed_mean_first50,months_over_cap_mean_first50,years_over_budget_mean_first50,gaps_under_cooldown_mean_first50
0,dev,M0_VF_logit,A_standard,circular >=7d,NaN,NaN,NaN,24.40,12.14,0.90,0.00
1,dev,M0_VF_logit,A_standard,regime-matched,NaN,NaN,NaN,36.18,17.76,2.02,101.72
2,dev,M0_VF_logit,A_standard,budget-preserving regime,0.9991,0.0009,0.0,0.00,0.00,0.00,0.00
3,dev,M0_VF_logit,B_pre_onset,circular >=7d,NaN,NaN,NaN,22.04,10.76,0.84,0.00
4,dev,M0_VF_logit,B_pre_onset,regime-matched,NaN,NaN,NaN,36.00,16.88,2.14,102.74
5,dev,M0_VF_logit,B_pre_onset,budget-preserving regime,0.9996,0.0004,0.0,0.00,0.00,0.00,0.00
6,dev,M0_VF_logit,B2_pre_onset_in_decline_negative,circular >=7d,NaN,NaN,NaN,21.42,10.70,0.96,0.00
7,dev,M0_VF_logit,B2_pre_onset_in_decline_negative,regime-matched,NaN,NaN,NaN,36.20,17.42,2.10,105.02
8,dev,M0_VF_logit,B2_pre_onset_in_decline_negative,budget-preserving regime,0.9996,0.0004,0.0,0.00,0.00,0.00,0.00
9,dev,M0_VF_logit,C_pre_peak,circular >=7d,NaN,NaN,NaN,16.78,8.26,0.74,0.00


pre  \
null                                                 circular >=7d   
period  set         label                                            
dev     M0_VF_logit A_standard                                22.0   
                    B_pre_onset                               20.0   
                    B2_pre_onset_in_decline_negative          20.0   
                    C_pre_peak                                15.0   
        M3_GBM_rich A_standard                                 7.0   
                    B_pre_onset                                5.0   
                    B2_pre_onset_in_decline_negative           7.0   
                    C_pre_peak                                 4.0   
holdout M0_VF_logit A_standard (notebook 11 stored)           13.0   

                                                                     \
null                                                 regime-matched   
period  set         label                                             
dev     M0_VF_logit A_standard                                 22.0   
                    B_pre_onset                                20.0   
                    B2_pre_onset_in_decline_negative           20.0   
                    C_pre_peak                                 15.0   
        M3_GBM_rich A_standard                                  7.0   
                    B_pre_onset                                 5.0   
                    B2_pre_onset_in_decline_negative            7.0   
                    C_pre_peak                                  4.0   
holdout M0_VF_logit A_standard (notebook 11 stored)            13.0   

                                                                               \
null                                                 budget-preserving regime   
period  set         label                                                       
dev     M0_VF_logit A_standard                                           22.0   
                    B_pre_onset                                          20.0   
                    B2_pre_onset_in_decline_negative                     20.0   
                    C_pre_peak                                           15.0   
        M3_GBM_rich A_standard                                            7.0   
                    B_pre_onset                                           5.0   
                    B2_pre_onset_in_decline_negative                      7.0   
                    C_pre_peak                                            4.0   
holdout M0_VF_logit A_standard (notebook 11 stored)                      13.0   

                                                     pre_excess_per_100_alerts  \
null                                                             circular >=7d   
period  set         label                                                        
dev     M0_VF_logit A_standard                                           2.157   
                    B_pre_onset                                          1.913   
                    B2_pre_onset_in_decline_negative                     1.908   
                    C_pre_peak                                           1.288   
        M3_GBM_rich A_standard                                           0.328   
                    B_pre_onset                                          0.200   
                    B2_pre_onset_in_decline_negative                     0.512   
                    C_pre_peak                                           0.090   
holdout M0_VF_logit A_standard (notebook 11 stored)                      0.248   

                                                                     \
null                                                 regime-matched   
period  set         label                                             
dev     M0_VF_logit A_standard                                2.007   
                    B_pre_onset                               1.789   
                    B2_pre_onset_in_decline_negative    

post  \
null                                                 circular >=7d   
period  set         label                                            
dev     M0_VF_logit A_standard                                17.0   
                    B_pre_onset                               18.0   
                    B2_pre_onset_in_decline_negative          18.0   
                    C_pre_peak                                12.0   
        M3_GBM_rich A_standard                                19.0   
                    B_pre_onset                               11.0   
                    B2_pre_onset_in_decline_negative          11.0   
                    C_pre_peak                                 6.0   
holdout M0_VF_logit A_standard (notebook 11 stored)           23.0   

                                                                     \
null                                                 regime-matched   
period  set         label                                             
dev     M0_VF_logit A_standard                                 17.0   
                    B_pre_onset                                18.0   
                    B2_pre_onset_in_decline_negative           18.0   
                    C_pre_peak                                 12.0   
        M3_GBM_rich A_standard                                 19.0   
                    B_pre_onset                                11.0   
                    B2_pre_onset_in_decline_negative           11.0   
                    C_pre_peak                                  6.0   
holdout M0_VF_logit A_standard (notebook 11 stored)            23.0   

                                                                               \
null                                                 budget-preserving regime   
period  set         label                                                       
dev     M0_VF_logit A_standard                                           17.0   
                    B_pre_onset                                          18.0   
                    B2_pre_onset_in_decline_negative                     18.0   
                    C_pre_peak                                           12.0   
        M3_GBM_rich A_standard                                           19.0   
                    B_pre_onset                                          11.0   
                    B2_pre_onset_in_decline_negative                     11.0   
                    C_pre_peak                                            6.0   
holdout M0_VF_logit A_standard (notebook 11 stored)                      23.0   

                                                     post_excess_per_100_alerts  \
null                                                              circular >=7d   
period  set         label                                                         
dev     M0_VF_logit A_standard                                            1.713   
                    B_pre_onset                                           1.825   
                    B2_pre_onset_in_decline_negative                      1.833   
                    C_pre_peak                                            1.088   
        M3_GBM_rich A_standard                                            2.044   
                    B_pre_onset                                           1.341   
                    B2_pre_onset_in_decline_negative                      1.313   
                    C_pre_peak                                            0.624   
holdout M0_VF_logit A_standard (notebook 11 stored)                       2.147   

                                                                     \
null                                                 regime-matched   
period  set         label                                             
dev     M0_VF_logit A_standard                                1.636   
                    B_pre_onset                               1.730   
                    B2_pre_onset_in_decline

## 5. Per-minute AUC: pre-onset and within-regime AUC by label, and differences from A

I score all variants on the same standard label (L1, 4%, event split); only the *training* label changes. All scores share the same bootstrap weights, so the intervals for differences from A are paired.

In [ ]:
code_all = np.where(Y == 1, np.where(INN, 2, 1), 0).astype(np.int64)
rngb = np.random.default_rng(SEED + 5000)
WTS = [np.bincount(rngb.integers(0, N_WEEKS, N_WEEKS), minlength=N_WEEKS).astype(float) for _ in range(N_BOOT)]
R21 = pd.read_csv(M21 / '1_onset_per_minute.csv')
PMR = []; BOOT = {}; t_ = time.time()
for mname in TRAINED:
    for li, lab in enumerate(LABELS):
        s = SC[(mname, lab)]; m = EVM & np.isfinite(s) & (DEC >= 0)
        P = make_parts(s[m], code_all[m], CALM[m], WEEK[m], DEC[m])
        point = stats_from(P, np.ones(N_WEEKS))
        if mname == 'M0_VF_logit' and lab == 'A_standard':
            from sklearn.metrics import roc_auc_score
            keep = code_all[m] != 2
            if abs(roc_auc_score((code_all[m][keep] > 0).astype(int), s[m][keep]) - point['auc_pre']) > 1e-9:
                raise RuntimeError('GUARD FAIL: block-pair AUC differs from sklearn')
        if lab == 'A_standard':
            ref = R21[(R21.onset == '1pct') & (R21.score == mname)]
            if len(ref) != 1 or any(abs(point[k] - float(ref.iloc[0][k])) > 1e-9 for k in ['auc_standard', 'auc_pre', 'in_share_of_excess']):
                raise RuntimeError(f'GUARD FAIL: {mname} A per-minute result differs from notebook 21')
        bt = {k: np.array([stats_from(P, w)[k] for w in WTS]) for k in STAT_KEYS}; BOOT[(mname, lab)] = bt
        row = dict(method=mname, label=lab, n_minutes=int(m.sum()), **point)
        for k in STAT_KEYS:
            row[f'{k}_lo'], row[f'{k}_hi'] = (float(x) for x in np.nanpercentile(bt[k], [2.5, 97.5]))
        PMR.append(row)
        print(f'{mname} {lab}: per-minute done ({time.time() - t_:.0f}s)', flush=True)
PM = pd.DataFrame(PMR)
for k in ['auc_standard', 'auc_pre', 'auc_pre_within_regime', 'in_share_of_excess', 'auc_calm']:
    PM[f'{k}_minus_A'] = np.nan; PM[f'{k}_minus_A_lo'] = np.nan; PM[f'{k}_minus_A_hi'] = np.nan
    for i, r in PM.iterrows():
        d = BOOT[(r.method, r.label)][k] - BOOT[(r.method, 'A_standard')][k]
        a0 = float(PM[(PM.method == r.method) & (PM.label == 'A_standard')][k].iloc[0])
        PM.loc[i, f'{k}_minus_A'] = r[k] - a0
        PM.loc[i, [f'{k}_minus_A_lo', f'{k}_minus_A_hi']] = np.nanpercentile(d, [2.5, 97.5])
PM.to_csv(OUT / '3_per_minute_by_label.csv', index=False)
print('PASS: A per-minute results equal notebook 21')
display(PM[['method', 'label', 'auc_standard', 'auc_pre', 'auc_in', 'in_share_of_excess', 'auc_pre_within_regime',
            'auc_pre_within_regime_minus_A', 'auc_pre_within_regime_minus_A_lo', 'auc_pre_within_regime_minus_A_hi']].round(3))

## 6. Reading with the pre-specified rules, figures and summary

In [ ]:
BP = 'budget-preserving regime'
dev = AL[(AL.period == 'dev') & (AL.null == BP)].set_index(['set', 'label'])
verdict = []
for mname in TRAINED:
    a_alerts = ALR[(mname, 'A_standard')]
    for lab in LABELS[1:]:
        r = PM[(PM.method == mname) & (PM.label == lab)].iloc[0]
        ex_x = float(dev.loc[(mname, lab), 'pre_excess_per_100_alerts']); ex_a = float(dev.loc[(mname, 'A_standard'), 'pre_excess_per_100_alerts'])
        cond_i = bool(r.auc_pre_within_regime_minus_A_lo > 0 and r.auc_pre_within_regime_minus_A >= MIN_AUC_GAIN); cond_ii = bool(ex_x - ex_a >= MIN_EXCESS_GAIN)
        v = 'improves' if (cond_i and cond_ii) else ('worse' if r.auc_pre_within_regime_minus_A_hi < 0 else 'no material improvement')
        pr = paired(EVD, a_alerts, ALR[(mname, lab)])
        verdict.append(dict(method=mname, label=lab, verdict=v, cond_i_within_regime_gain=cond_i, cond_ii_excess_gain=cond_ii,
                            within_regime_pre_auc=float(r.auc_pre_within_regime), within_regime_pre_auc_A=float(r.auc_pre_within_regime - r.auc_pre_within_regime_minus_A),
                            diff=float(r.auc_pre_within_regime_minus_A), diff_lo=float(r.auc_pre_within_regime_minus_A_lo), diff_hi=float(r.auc_pre_within_regime_minus_A_hi),
                            pre_excess_budget=ex_x, pre_excess_budget_A=ex_a,
                            pre=int(dev.loc[(mname, lab), 'pre']), pre_A=int(dev.loc[(mname, 'A_standard'), 'pre']),
                            post=int(dev.loc[(mname, lab), 'post']), post_A=int(dev.loc[(mname, 'A_standard'), 'post']),
                            mcnemar_pre_new_better_p=pr['p_pre1pct_one_sided_new_better'], new_only_pre=pr['new_only_pre1pct'], base_only_pre=pr['base_only_pre1pct']))
VD = pd.DataFrame(verdict); VD.to_csv(OUT / '4_reading_rule_verdicts.csv', index=False); display(VD.round(4))

m0d = AL[(AL.period == 'dev') & (AL.set == 'M0_VF_logit') & (AL.label == 'A_standard')].set_index('null')
m0h = AL[AL.period == 'holdout'].set_index('null')
robust_dev = bool(m0d.loc[BP, 'p_pre'] < 0.05); holdout_unchanged = bool(m0h.loc[BP, 'p_pre'] >= 0.05)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6)); x = np.arange(len(LABELS)); short = ['A', 'B', 'B2', 'C']
for k, (mname, colr) in enumerate([('M0_VF_logit', '#2471a3'), ('M3_GBM_rich', '#7f8c8d')]):
    g = PM[PM.method == mname].set_index('label').loc[LABELS]
    ax[0].errorbar(x + (k - 0.5) * 0.2, g.auc_pre_within_regime, yerr=[g.auc_pre_within_regime - g.auc_pre_within_regime_lo,
                   g.auc_pre_within_regime_hi - g.auc_pre_within_regime], fmt='o', capsize=4, color=colr, label=mname)
    gd = dev.loc[mname].loc[LABELS]
    ax[1].bar(x + (k - 0.5) * 0.4, gd.pre_excess_per_100_alerts, 0.4, color=colr, label=mname)
ax[0].axhline(0.5, color='k', lw=0.8, ls=':'); ax[0].set_xticks(x); ax[0].set_xticklabels(short)
ax[0].set_title('Within-regime pre-onset AUC by training label\n(dev, 95% block-bootstrap CI)', fontsize=9); ax[0].legend(fontsize=8)
ax[1].axhline(0, color='k', lw=0.8); ax[1].set_xticks(x); ax[1].set_xticklabels(short)
ax[1].set_title('Above-chance pre-onset catches per 100 alerts\n(budget-preserving regime null)', fontsize=9); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / '5_onset_aware_training.png', dpi=140); plt.show()

summary = dict(spec_sha256=SPEC_SHA, frozen_sha256=FROZEN_SHA, status='exploratory; dev 2023-01..2026-04; holdout = stored notebook-11 alerts only',
               reading_rule=SPEC['reading_rule'], null_audit_rule=SPEC['null_audit_rule'],
               verdicts=VD.to_dict('records'), dev_m0_pre_robust_to_budget_null=robust_dev, holdout_pre_reading_unchanged=holdout_unchanged,
               training_label_counts=LC.to_dict('records'), null_constraint_audit=AU.to_dict('records'),
               alerts=AL.to_dict('records'),
               per_minute=PM[['method', 'label', 'auc_standard', 'auc_pre', 'auc_in', 'share_in', 'in_share_of_excess', 'auc_pre_within_regime',
                              'auc_pre_within_regime_lo', 'auc_pre_within_regime_hi', 'auc_pre_within_regime_minus_A', 'auc_pre_within_regime_minus_A_lo',
                              'auc_pre_within_regime_minus_A_hi', 'auc_pre_minus_A', 'auc_pre_minus_A_lo', 'auc_pre_minus_A_hi']].to_dict('records'))
(OUT / 'ONSET_AWARE_SUMMARY.json').write_text(json.dumps(summary, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))

for r in VD.itertuples():
    print(f"{r.method:12s} {r.label:34s} -> {r.verdict:23s} | within-regime pre-onset AUC {r.within_regime_pre_auc:.3f} vs A {r.within_regime_pre_auc_A:.3f} "
          f"(diff {r.diff:+.3f} [{r.diff_lo:+.3f}, {r.diff_hi:+.3f}]) | pre-onset excess/100 alerts {r.pre_excess_budget:+.2f} vs A {r.pre_excess_budget_A:+.2f} "
          f"| pre {r.pre} vs {r.pre_A}, in-progress {r.post} vs {r.post_A}")
for tag, g in [('dev M0', m0d), ('holdout M0 (stored)', m0h)]:
    print(f"{tag:20s} pre-onset {int(g.loc[BP, 'pre'])}: null means / p — circular {g.loc['circular >=7d', 'pre_null_mean']:.2f} / {g.loc['circular >=7d', 'p_pre']:.4f}, "
          f"regime {g.loc['regime-matched', 'pre_null_mean']:.2f} / {g.loc['regime-matched', 'p_pre']:.4f}, budget-preserving {g.loc[BP, 'pre_null_mean']:.2f} / {g.loc[BP, 'p_pre']:.4f}")
print(f"Null audit rule: dev pre-onset robust to the budget constraint = {robust_dev}; holdout reading unchanged = {holdout_unchanged}")

## Interpretation
- The main results are the per-label verdicts (`improves` / `no material improvement` / `worse`; `4_reading_rule_verdicts.csv`) and the two null-audit lines (`ONSET_AWARE_SUMMARY.json`).
- If B and C show "no material improvement": aligning the training target with early warning adds no timing information beyond the regime with the same features. This supports the conclusion that training and evaluation reward detection.
- If a label "improves": this is a stronger finding that the standard label is unsuited to early warning, but it is not confirmatory because I have already examined the development period; I keep the frozen A, B, B2 and C models (`ONSET_AWARE_FROZEN.json`) for the preregistered test.
- B and C did not learn in-progress stretches as positives, so I expect in-progress detection to fall. If it does not, in-progress detection comes from the features (volatility) rather than from the label.
- In `2_null_constraint_audit.csv` I report whether the budget-preserving null leaves monthly counts unchanged with no gaps under 240 minutes, and the share of alerts placed in the same decile.